## Importing libraries + reading data from csv format into pandas.DataFrame

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
from pathlib import Path

DATA_PATH = Path("..") / "data" / "raw"

df_air_reserve = pd.read_csv(DATA_PATH / "air_reserve.csv")
df_air_store_info = pd.read_csv(DATA_PATH / "air_store_info.csv")
df_air_visit_data = pd.read_csv(DATA_PATH / "air_visit_data.csv")
df_date_info = pd.read_csv(DATA_PATH / "date_info.csv")
df_hpg_reserve = pd.read_csv(DATA_PATH / "hpg_reserve.csv")
df_hpg_store_info = pd.read_csv(DATA_PATH / "hpg_store_info.csv")
df_store_id_relation = pd.read_csv(DATA_PATH / "store_id_relation.csv")

In [ ]:
data_frames = [
        df_air_reserve, df_air_store_info, df_air_visit_data, df_date_info,
        df_hpg_reserve, df_hpg_store_info, df_store_id_relation
    ]

data_frames_names = [
    "air_reserve", "air_store_info", "air_visit_data", "date_info",
    "hpg_reserve", "hpg_store_info", "store_id_relation"
]

## Counting nan values and unique values in the dataframes

In [ ]:
from restaurant_visitor_eda.dataset import count_unique_and_nans

In [ ]:
for i, df in enumerate(data_frames):
    print(data_frames_names[i])
    display(count_unique_and_nans(df))
    #print(count_unique_and_nans(df))

## Conclusion: 

1) There are __no__ nan values in the data set, however, this does __not__ particularly mean that the data set is complete or consistent. The next points will look into my arguments in favour of this position

2) There $829$ unique restaurants in the air_store_info table, while only $314$ are covered in air_reserve. This means that we do not have any information about reserves made in $(829 - 314) = 515$ restaurants through air service.

3) The store_id_relation has only $150$ rows. Thus, we can join information from two services only for $150$ places, which is significantly smaller than the total number of the restaraunts in the area


## Target Value 

In [ ]:
from restaurant_visitor_eda.plots import plot_target_distribution

plot_target_distribution(df_air_visit_data)

In [ ]:
from restaurant_visitor_eda.dataset import get_stats

get_stats(pd.DataFrame(df_air_visit_data['visitors']))

In [ ]:
get_stats(pd.DataFrame(np.log1p(df_air_visit_data['visitors'])))

## Conclusion!!!


## Features Analysis



### Restaurants' genres


#### Distribution of the geners


In [ ]:
from restaurant_visitor_eda.plots import build_barplot_for_air_genres
build_barplot_for_air_genres(df_air_store_info)

In [ ]:
from restaurant_visitor_eda.plots import build_barplot_for_hpg_genres
build_barplot_for_hpg_genres(df_hpg_store_info)

In [ ]:
from restaurant_visitor_eda.plots import plot_median_visitors_per_genre, plot_visitors_boxplot_air
plot_median_visitors_per_genre(pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
plot_visitors_boxplot_air(pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
df_air_store_info[['prefecture', 'district', 'block']] = df_air_store_info['air_area_name'].str.strip().str.split(' ', n=2, expand=True)


In [ ]:
from restaurant_visitor_eda.plots import build_barplot_for_air_prefectures, build_barplot_for_air_districts, build_barplot_for_air_blocks
build_barplot_for_air_prefectures(data=df_air_store_info)
build_barplot_for_air_districts(data=df_air_store_info)
build_barplot_for_air_blocks(data=df_air_store_info)

In [ ]:
from restaurant_visitor_eda.plots import plot_median_visitors_per_prefecture

plot_median_visitors_per_prefecture(df=pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
from restaurant_visitor_eda.plots import plot_median_visitors_per_district

plot_median_visitors_per_district(df=pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
from restaurant_visitor_eda.plots import plot_median_visitors_per_block

plot_median_visitors_per_block(df=pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
from restaurant_visitor_eda.plots import plot_genre_prefecture_heatmap

plot_genre_prefecture_heatmap(df=df_air_store_info)

In [ ]:
from restaurant_visitor_eda.plots import plot_visitors_heatmap

plot_visitors_heatmap(pd.merge(df_air_visit_data, df_air_store_info, on='air_store_id'))

In [ ]:
df_air_visit_data['visit_datetime'] = pd.to_datetime(df_air_visit_data['visit_date'])

In [ ]:
df_date_info['date'] = pd.to_datetime(df_date_info['calendar_date'])
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_date_info['day_of_week'] = pd.Categorical(df_date_info['day_of_week'], categories=days_order, ordered=True)

In [ ]:
df_joined_for_day_of_week_holidays = pd.merge(df_air_visit_data, df_date_info, left_on='visit_datetime', right_on='date')
df_joined_for_day_of_week_holidays

In [ ]:
from restaurant_visitor_eda.plots import plot_visitors_boxplot_air_by_day

plot_visitors_boxplot_air_by_day(df_joined_for_day_of_week_holidays)

In [ ]:
from restaurant_visitor_eda.plots import plot_visitors_boxplot_air_by_holiday

plot_visitors_boxplot_air_by_holiday(df_joined_for_day_of_week_holidays)

In [ ]:
from restaurant_visitor_eda.plots import plot_visitors_boxplot_air_by_holiday_and_day

plot_visitors_boxplot_air_by_holiday_and_day(df_joined_for_day_of_week_holidays)

In [ ]:
from restaurant_visitor_eda.dataset import calculate_holiday_significance

calculate_holiday_significance(df_joined_for_day_of_week_holidays)

In [ ]:
df_joined_for_day_of_week_holidays.sort_values(by='date', ascending=True, inplace=True)

In [ ]:
df_joined_for_day_of_week_holidays

In [ ]:
import numpy as np

df_date_info = df_date_info.sort_values('calendar_date')
df_date_info['next_is_holiday'] = df_date_info['holiday_flg'].shift(-1).fillna(0).astype(int)

conditions = [
    (df_date_info['holiday_flg'] == 1) & (df_date_info['next_is_holiday'] == 1),
    (df_date_info['holiday_flg'] == 1) & (df_date_info['next_is_holiday'] == 0),
    (df_date_info['holiday_flg'] == 0) & (df_date_info['next_is_holiday'] == 1), 
    (df_date_info['holiday_flg'] == 0) & (df_date_info['next_is_holiday'] == 0)  
]

choices = [
    'Holiday & Holiday Tomorrow', 
    'Holiday & Workday Tomorrow',
    'Workday & Holiday Tomorrow (Eve)', 
    'Workday & Workday Tomorrow'
]

df_date_info['day_pattern'] = np.select(conditions, choices, default='Unknown')

df_final = pd.merge(
    df_air_visit_data, 
    df_date_info[['calendar_date', 'day_pattern', 'holiday_flg', 'day_of_week']], 
    left_on='visit_date', 
    right_on='calendar_date'
)

In [ ]:
from restaurant_visitor_eda.plots import plot_visitors_boxplot_air_by_holiday_and_day_and_eve

plot_visitors_boxplot_air_by_holiday_and_day_and_eve(df_final)